In [1]:
# import useful lib
import numpy as np
import time
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

client = RemoteAPIClient()
sim = client.getObject('sim')

# Run a simulation in asynchronous Mode
client.setStepping(False)
sim.startSimulation()

print("Connected to CoppeliaSim and simulation started!")

Connected to CoppeliaSim and simulation started!


In [2]:
# Forward Kinematics Function:

def FK(theta, alpha, a, d):
    
    #conversion of theta to radians
    theta = theta * np.pi/180
    
    #conversion of alpha to radians
    alpha = alpha * np.pi/180
    
    Cth = np.cos(theta)  #Cos_theta
    Sth = np.sin(theta)  #Sin_theta
    Calp = np.cos(alpha) #Cos_alpha
    Salp = np.sin(alpha) #Sin_alpha
    
    #Forward Transformation Matrix from the DH-Table
    T = np.array([
                  [Cth, -Sth*Calp,  Sth*Salp,  a*Cth],
                  [Sth,  Cth*Calp, -Cth*Salp,  a*Sth],
                  [0,    Salp,      Calp,      d],
                  [0,    0,         0,         1]
    ])
    
    return np.round(T, 3)

In [3]:
q = [0, 0, 0, 0, 0, 0, 0]

In [4]:
# Transformations of DH table for the KUKA iiwa 7 R800

T0_0 = FK(0, 0, 0, 0)
T0_1 = FK(q[0], -90, 0, 0.34 )
T1_2 = FK(q[1],  90, 0, 0    )
T2_3 = FK(q[2],  90, 0, 0.4  )
T3_4 = FK(q[3], -90, 0, 0    )
T4_5 = FK(q[4], -90, 0, 0.4  )
T5_6 = FK(q[5],  90, 0, 0    )
T6_7 = FK(q[6],   0, 0, 0.126)

# Get Transformation from end-effector frame to base frame

T0_2 = T0_1 @ T1_2
T0_3 = T0_1 @ T1_2 @ T2_3
T0_4 = T0_1 @ T1_2 @ T2_3 @ T3_4
T0_5 = T0_1 @ T1_2 @ T2_3 @ T3_4 @ T4_5
T0_6 = T0_1 @ T1_2 @ T2_3 @ T3_4 @ T4_5 @ T5_6

T0_7 = T0_1 @ T1_2 @ T2_3 @ T3_4 @ T4_5 @ T5_6 @ T6_7

print("Analytical (T0_7) = \n", np.round(T0_7, 2))

Analytical (T0_7) = 
 [[1.   0.   0.   0.  ]
 [0.   1.   0.   0.  ]
 [0.   0.   1.   1.27]
 [0.   0.   0.   1.  ]]


In [5]:
# Getting the joints names:
joint_names = [f'joint_{i+1}' for i in range(7)]

print ("Joint Names are: ", joint_names)

Joint Names are:  ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 'joint_6', 'joint_7']


In [6]:
# Getting the joints Handels:

#######################################
# Base    = sim.getObjectHandle("joint_1")
# joint_1 = sim.getObjectHandle("joint_1")
# joint_2 = sim.getObjectHandle("joint_2")
# joint_3 = sim.getObjectHandle("joint_3")
# joint_4 = sim.getObjectHandle("joint_4")
# joint_5 = sim.getObjectHandle("joint_5")
# joint_6 = sim.getObjectHandle("joint_6")
# joint_7 = sim.getObjectHandle("joint_7")
# EE      = sim.getObjectHandle("joint_7")
#######################################

# Getting the joints Handels:
joint_handles=[sim.getObjectHandle(name) for name in joint_names]

# Setting both of the End Effector & Base Handles:
Base = sim.getObjectHandle("joint_1")
EE = sim.getObjectHandle("joint_7")

EE_Dummy = sim.getObjectHandle("EE_Dummy")

print("Joint Handles are: ", joint_handles)

Joint Handles are:  [18, 21, 24, 27, 30, 33, 36]


In [7]:
# Enable the position control for all joints
for i in range(len(joint_handles)):
    sim.setObjectInt32Parameter(joint_handles[i], 2001, ~0)

In [8]:
sim.startSimulation()

#Analytical:
print("T0_7")
print("Analytical Transformation of the End Effector\n", np.round(T0_7, 2))

print("============================ \n")

#Actual
for i in range(7):
    sim.setJointTargetPosition(joint_handles[i],q[i]*np.pi/180)

time.sleep(0.5)

T_EE = sim.getObjectMatrix(EE_Dummy,-1)

# Convert the flattened 3x4 matrix into a 4x4 matrix
T_EE = np.array(T_EE).reshape(3, 4)      # Reshape as 3x4
T_EE = np.vstack((T_EE, [0, 0, 0, 1]))   # Add the last row [0, 0, 0, 1]

print("Actual Transformation of the End Effector\n", np.round(T_EE, 2))

T0_7
Analytical Transformation of the End Effector
 [[1.   0.   0.   0.  ]
 [0.   1.   0.   0.  ]
 [0.   0.   1.   1.27]
 [0.   0.   0.   1.  ]]

Actual Transformation of the End Effector
 [[ 1.   -0.    0.    0.01]
 [ 0.    1.   -0.    0.  ]
 [-0.    0.    1.    1.21]
 [ 0.    0.    0.    1.  ]]


In [9]:
#Jacobian Function

def Jacobian(T0, T1, T2, T3, T4, T5, T6, T7):
    linksToZero = [T0, T1, T2, T3, T4, T5, T6, T7]
    Jp=np.array([[], [], []])
    Jr=np.array([[], [], []])
    
    Pos_T07=np.array([T7[0][3], T7[1][3], T7[2][3]])
    
    for i in range(7):
        R = np.array([linksToZero[i][0][2], linksToZero[i][1][2], linksToZero[i][2][2]])
        P = np.array([linksToZero[i][0][3], linksToZero[i][1][3], linksToZero[i][2][3]])
        
        Psubtract=np.subtract(Pos_T07, P)
        
        Jpp=np.cross(R, Psubtract)
        Jp=np.hstack((Jp, np.atleast_2d(Jpp).T))
        Jr=np.hstack((Jr, np.atleast_2d(R).T))
        
    J=np.concatenate((Jp, Jr))
    return J

In [10]:
# Inverse Kinematics Function
def deltaX(T7, RD, PD):
    # Create a 3x3 identity matrix
    I = np.eye(3)
    
    # Extract the 3x3 rotation matrix from the transformation matrix (T7)
    R0 = np.array([T7[0][0:3], T7[1][0:3], T7[2][0:3]])
    
    # Extract the position matrix from the transformation matrix (T7)
    P0 = np.array([T7[0][3], T7[1][3], T7[2][3]])
    
    # Compute the transpose of the current rotation matrix (R0)
    R0T = R0.transpose()
    
    # Compute the rotation difference matrix: RD * R0T - I
    # RD is the desired rotation matrix, R0 is the current rotation matrix
    skew_w = RD @ R0T - I
    
    # Extract the rotation error from the skew matrix
    vec_w = np.array([[skew_w[2][1]], [skew_w[0][2]], [skew_w[1][0]]])
    
    # Compute the difference between desired (PD) and current positions (P0)
    deltax = np.array([PD[0] - P0[0], PD[1] - P0[1], PD[2] - P0[2]])
    
    # Concatenate the positional difference (deltax) and angular difference (vec_w)
    # The result is a 6x1 array containing both linear and angular errors
    deltax = np.concatenate((deltax, vec_w))
    
    # Return the combined error (linear + angular)
    return deltax

In [11]:
# Inverse Kinematics Function
def IK(J, deltax, Q0):
    # Compute the pseudo-inverse of the Jacobian matrix (J)
    # This is used to map the 6x1 end-effector error vector (deltax) to joint space
    Jplus = np.round(np.linalg.pinv(J), 2) # Round to 2 decimal places for stability

    # Compute the joint angle increments (delta Q)
    # This is the product of the pseudo-inverse Jacobian and the error vector
    JJ = Jplus @ deltax

    # Update the current joint angles (Q0) by adding the increments
    Qd = Q0 + JJ  # # Update joint angles

    # Return the updated joint angles
    return Qd

In [ ]:
# Example for a desired Position:
PosD = np.array([[-0.872], [+0.42452], [+0.00515]])

# Use a valid desired rotation matrix for the end effector.

RotD = np.eye(3)

# initial joint angles
Q0 = np.array([[90], [0], [-90], [0], [90], [-180], [0]], dtype=float)

# Helper to compute all cumulative base-frame transforms for the current joint vector

def compute_T0_frames(q):
    T0_1 = FK(q[0, 0], -90, 0, 0.34)
    T1_2 = FK(q[1, 0], 90, 0, 0)
    T2_3 = FK(q[2, 0], 90, 0, 0.4)
    T3_4 = FK(q[3, 0], -90, 0, 0)
    T4_5 = FK(q[4, 0], -90, 0, 0.4)
    T5_6 = FK(q[5, 0], 90, 0, 0)
    T6_7 = FK(q[6, 0], 0, 0, 0.126)

    T0_2 = T0_1 @ T1_2
    T0_3 = T0_2 @ T2_3
    T0_4 = T0_3 @ T3_4
    T0_5 = T0_4 @ T4_5
    T0_6 = T0_5 @ T5_6
    T0_7 = T0_6 @ T6_7

    return T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7

T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7 = compute_T0_frames(Q0)
Jacob = Jacobian(T0_0, T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7)
delX = deltaX(T0_7, RotD, PosD)

q = Q0.copy()
normdelX = np.linalg.norm(delX)
iteration = 0
max_iter = 200

while normdelX > 0.01 and iteration < max_iter:
    q = IK(Jacob, delX, q)
    T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7 = compute_T0_frames(q)
    Jacob = Jacobian(T0_0, T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7)
    delX = deltaX(T0_7, RotD, PosD)
    normdelX = np.linalg.norm(delX)
    iteration += 1

print(f"IK finished after {iteration} iterations, normdelX={normdelX:.4f}")
print("Final joint angles (degrees):", q.T)


IK finished after 100 iterations, normdelX=1.5391
Final joint angles (degrees): [[-2.44927287e+01 -2.52257569e+03  4.88263025e-01 -5.14738782e+03
   1.13319941e+00 -2.62481783e+03  2.70543392e+01]]


In [13]:
# Calculate the Jacobian matrix for the current joint configuration
# T0_0 to T0_7 are the transformation matrices from the base frame to each joint or end-effector frame
Jacob = Jacobian(T0_0, T0_1, T0_2, T0_3, T0_4, T0_5, T0_6, T0_7)

# Calculate the error vector (deltaX) between the desired and current end-effector pose
# T0_7 is the current end-effector transformation matrix
# RotD is the desired rotation matrix
# PosD is the desired position
delX = deltaX(T0_7, RotD, PosD)

# Calculate the norm (magnitude) of the error vector (delX)
# This measures how far the end-effector is from the desired pose
normdelX = np.linalg.norm(delX)
print('Norm Delta X\n', normdelX)

# Round the computed joint angles (q) to two decimal places for numerical stability
q = np.round(q, 2)
print('Q desired:\n', q.item((0, 0)))

# Loop through all 7 joints to set the target positions in the simulator
# Converts the joint angles from degrees to radians and sets them in CoppeliaSim
for i in range(7):
    sim.setJointTargetPosition(joint_handles[i], float(q[i, 0]) * np.pi / 180)


Norm Delta X
 1.5391041119410134
Q desired:
 -24.49
